# Первичный анализ

In [2]:
import pandas as pd

base = "https://python-academy.org/static/product-analytics/dataset"


# parse_dates преобразует столбцы в тип даты для датафрейма
users = pd.read_csv(f"{base}/users.csv", parse_dates=["signup_date"])
products = pd.read_csv(f"{base}/products.csv")
sessions = pd.read_csv(f"{base}/sessions.csv", parse_dates=["started_at"])
orders = pd.read_csv(f"{base}/orders.csv", parse_dates=["created_at"])

print(f"users:    {len(users):>7,}")

print(f"products: {len(products):>7,}")

print(f"sessions: {len(sessions):>7,}")

print(f"orders:   {len(orders):>7,}")

users:      6,000
products:     300
sessions:  68,668
orders:     7,327


In [5]:
# Посмотреть тип данных колонок в датафрейме
users.dtypes

user_id                     object
signup_date         datetime64[ns]
source                      object
country                     object
device_first                object
age_bucket                  object
marketing_opt_in              bool
dtype: object

In [7]:
# Посмотреть историчность данных во всех таблицах
print(f"users:    {users['signup_date'].min().date()} → {users['signup_date'].max().date()}")

print(f"sessions: {sessions['started_at'].min().date()} → {sessions['started_at'].max().date()}")

print(f"orders:   {orders['created_at'].min().date()} → {orders['created_at'].max().date()}")

users:    2025-06-01 → 2026-05-31
sessions: 2025-06-01 → 2026-05-31
orders:   2025-06-01 → 2026-06-01


In [12]:
# Проверка пропусков
print(users.isna().sum())

user_id               0
signup_date           0
source                0
country               0
device_first          0
age_bucket          309
marketing_opt_in      0
dtype: int64


In [13]:
print(sessions.isna().sum())

session_id         0
user_id            0
started_at         0
duration_sec       0
device             0
source             0
country         2078
pages_viewed       0
reached_step       0
dtype: int64


In [25]:
# Уникальные значения в колонке
print(users["source"].unique())

['paid_social' 'organic' 'paid_search' 'referral' 'email']


In [14]:
# Проверка дублей
print(f"дублей user_id:    {users['user_id'].duplicated().sum()}")

print(f"дублей session_id: {sessions['session_id'].duplicated().sum()}")

print(f"дублей order_id:   {orders['order_id'].duplicated().sum()}")

дублей user_id:    0
дублей session_id: 8
дублей order_id:   0


In [17]:
# Описание базовой статистики конкретного столбца

# Среднее в полтора раза больше медианы, а максимум в пять раз выше 75-го перцентиля — 
# это намёк на длинный хвост: небольшая доля крупных заказов сильно тянет среднее вверх.
print(orders["total_rub"].describe())

count      7327.000000
mean      34012.965743
std       34653.869367
min         230.000000
25%        8650.000000
50%       21900.000000
75%       47100.000000
max      243400.000000
Name: total_rub, dtype: float64


# Срезы и фильтры

In [19]:
# Конструкция-аналог where

completed = orders[orders["status"] == "completed"]
print(f"завершённых заказов: {len(completed)} из {len(orders)}")

завершённых заказов: 6986 из 7327


In [21]:
# Применение метода к маске считается автоматически по True условию
# Одна строка вместо двух подсчётов и деления — так доли считают в реальной работе

print(f"доля завершённых: {(orders['status'] == 'completed').mean():.1%}")

print(f"доля мобильных сессий: {(sessions['device'] == 'mobile').mean():.1%}")

доля завершённых: 95.3%
доля мобильных сессий: 57.9%


In [24]:
# Конструкция-аналог where с несколькими условиями
# Кроме & есть | (или) и ~ (не)

march = orders[
    (orders["created_at"] >= "2026-03-01") & (orders["created_at"] < "2026-04-01")
]
print(f"заказов в марте: {len(march)}")

заказов в марте: 1025


In [26]:
# Конструкция-аналог in

cis = users[users["country"].isin(["KZ", "BY", "UA"])]
print(f"пользователей из KZ, BY, UA: {len(cis)}")

пользователей из KZ, BY, UA: 1835


In [28]:
# в query можно через строку вставить много условий
# имена колонок без кавычек, а строковые значения — в одинарных кавычках

big = orders.query("total_rub > 50_000 and status == 'completed'")
print(f"крупных завершённых заказов: {len(big)}")

крупных завершённых заказов: 1639


In [30]:
# loc для фильтрации только нужных столбцов датафрейма

kz = users.loc[users["country"] == "KZ", ["user_id", "signup_date", "source"]]
print(f"пользователей из KZ: {len(kz)}")

print(kz.head())

пользователей из KZ: 921
    user_id signup_date       source
0   u_00001  2026-04-04  paid_social
18  u_00019  2026-04-18  paid_search
25  u_00026  2025-09-17     referral
29  u_00030  2026-03-11      organic
31  u_00032  2026-05-23  paid_social


In [31]:
# Для создания нового отфильтрованного датафрейма стоит создать копию среза

kz = users.loc[users["country"] == "KZ", ["user_id", "signup_date", "source"]].copy()

# Группировки

In [3]:
# Сколько строк в каждой группе

print(users["source"].value_counts())

source
paid_social    2082
organic        1502
paid_search    1243
email           594
referral        579
Name: count, dtype: int64


In [5]:
# С аргументом normalize подсчет доли вместо количества

print((users["source"].value_counts(normalize=True) * 100).round(1))

source
paid_social    34.7
organic        25.0
paid_search    20.7
email           9.9
referral        9.6
Name: proportion, dtype: float64


In [6]:
# groupby среднее total_rub по полю items_count

print(orders.groupby("items_count")["total_rub"].mean().round(0))

items_count
1    16739.0
2    33888.0
3    51248.0
Name: total_rub, dtype: float64


In [7]:
# agg позволяет считать несколько разных агрегаций, как в SQL через запятую

print(orders.groupby("status").agg(
    orders=("order_id", "count"),     # имя_колонки=(откуда, чем считать)
    revenue=("total_rub", "sum"),
    avg_order=("total_rub", "mean"),
).round(0))

           orders    revenue  avg_order
status                                 
cancelled     205    7136750    34813.0
completed    6986  236572460    33864.0
returned      136    5503790    40469.0


In [9]:
# size() считает строки в каждой группе, как value_counts

print(orders.groupby(orders["created_at"].dt.to_period("M")).size())

created_at
2025-06      13
2025-07      45
2025-08      90
2025-09     136
2025-10     230
2025-11     384
2025-12     468
2026-01     576
2026-02     664
2026-03    1025
2026-04    1267
2026-05    2427
2026-06       2
Freq: M, dtype: int64


In [19]:
# loc[source] позволяет вывести только нужную/нужные строки

source = 'organic'
res = (users['source'].value_counts(normalize=True) * 100).round(1).loc[source]
print(res)

25.0


# Объединение таблиц

In [21]:
# merge аналоги join

merged = orders.merge(users[["user_id", "country"]], on="user_id")

# проверка строк и столбцов датафрейма
print(f"после: {merged.shape}")

print(merged[["order_id", "user_id", "total_rub", "status", "country"]].head())

после: (7327, 8)
  order_id  user_id  total_rub     status country
0  o_00001  u_00002      25150  completed      RU
1  o_00002  u_00003      31650  completed      RU
2  o_00003  u_00009      13900  completed      UA
3  o_00004  u_00010        690  completed      RU
4  o_00005  u_00013       2100  completed      US


In [23]:
# Сначала фильтруем ранее сджойненный датафрейм, потом считает группировки

completed = merged[merged["status"] == "completed"]
revenue = completed.groupby("country")["total_rub"].sum().sort_values(ascending=False)
print(revenue)

country
RU    118048670
KZ     36474780
BY     22779310
UA     12280270
US     12220350
DE     10534790
PL      7401090
TR      7242150
AM      5434320
GE      4156730
Name: total_rub, dtype: int64


In [24]:
# Перевод результата в миллионы

print((revenue / 1e6).round(1))

country
RU    118.0
KZ     36.5
BY     22.8
UA     12.3
US     12.2
DE     10.5
PL      7.4
TR      7.2
AM      5.4
GE      4.2
Name: total_rub, dtype: float64


In [25]:
# Аналог left join

left = users.merge(orders[["user_id", "order_id"]], on="user_id", how="left")

# Проверка сколько строк с null в конкретном столбце
print(f"пользователей без заказов: {left['order_id'].isna().sum()}")

пользователей без заказов: 2303


In [27]:
# nunique аналог distinct count

buyers = orders["user_id"].nunique()
print(f"покупателей: {buyers} из {len(users)} ({buyers / len(users):.1%})")

покупателей: 3697 из 6000 (61.6%)


# Работа с датами

In [30]:
print("месяц:", orders["created_at"].dt.month.head(3).tolist())

print("год:", orders["created_at"].dt.year.head(3).tolist())

print("день недели:", orders["created_at"].dt.day_name().head(3).tolist())


print("Период год-месяц:", orders["created_at"].dt.to_period('M').head(3).tolist())

print("Период год:", orders["created_at"].dt.to_period('Y').head(3).tolist())

print("Период день", orders["created_at"].dt.to_period('D').head(3).tolist())


месяц: [12, 5, 11]
год: [2025, 2026, 2025]
день недели: ['Monday', 'Saturday', 'Friday']
Период год-месяц: [Period('2025-12', 'M'), Period('2026-05', 'M'), Period('2025-11', 'M')]
Период год: [Period('2025', 'Y-DEC'), Period('2026', 'Y-DEC'), Period('2025', 'Y-DEC')]
Период день [Period('2025-12-22', 'D'), Period('2026-05-09', 'D'), Period('2025-11-21', 'D')]


In [32]:
# resample заполнит нулями столбец, если по этому значению нет данных
# шаг сетки задаётся строкой: "D" дни, "W" недели, "ME" месяцы.

daily = orders.set_index("created_at").resample("D").size()

# внутри loc можно указывать диапазон как в списке
print(daily.loc["2025-11-24":"2025-12-03"])

created_at
2025-11-24     9
2025-11-25    10
2025-11-26    15
2025-11-27    17
2025-11-28    25
2025-11-29    26
2025-11-30    25
2025-12-01    17
2025-12-02    10
2025-12-03    13
Freq: D, dtype: int64


In [34]:
# выручка по месяцам в миллионах

completed = orders[orders["status"] == "completed"]
monthly = completed.set_index("created_at")["total_rub"].resample("ME").sum()
print((monthly / 1e6).round(1))

created_at
2025-06-30     0.3
2025-07-31     1.1
2025-08-31     3.4
2025-09-30     4.3
2025-10-31     8.9
2025-11-30    12.5
2025-12-31    15.5
2026-01-31    18.0
2026-02-28    20.7
2026-03-31    31.2
2026-04-30    40.6
2026-05-31    80.0
2026-06-30     0.0
Freq: ME, Name: total_rub, dtype: float64


In [35]:
# Сколько заказов приходится на сто сессий каждый месяц?

completed = orders[orders["status"] == "completed"]
monthly = completed.set_index("created_at")["total_rub"].resample("ME").sum()
print((monthly / 1e6).round(1))

created_at
2025-06-30     0.3
2025-07-31     1.1
2025-08-31     3.4
2025-09-30     4.3
2025-10-31     8.9
2025-11-30    12.5
2025-12-31    15.5
2026-01-31    18.0
2026-02-28    20.7
2026-03-31    31.2
2026-04-30    40.6
2026-05-31    80.0
2026-06-30     0.0
Freq: ME, Name: total_rub, dtype: float64
